# Week 6 - Tensor Cores and MMA Programming (Guide)

This notebook is a practical concise execution guide for **Mini-Project 6**. It is intentionally written as a workflow, not just a concept summary: what to implement, how to measure, how to validate, what can break, and how to report results. I am assuming at this point in the course you have no idea what tensorcores are, or your understanding is limited. I encourage you to watch this video before you start, it will look very confusing, but just watch it at the start and at the end (trust me) [What are tensorcores](https://www.youtube.com/watch?v=Cak8ASX7NOk).

## Mini-Project Target
Implement Tensor Core GEMM using WMMA for FP16 and TF32, then evaluate:

1. Throughput (TFLOPS achieved).
2. Numerical error vs FP32 cuBLAS baseline.
3. Loss-scaling impact on tiny gradient survival.

## Why This Project Matters

Modern training stacks rely on mixed precision because Tensor Cores massively increase arithmetic density. That speedup is only useful if you understand where precision is lost and how to keep training stable.

The practical debugging questions this lab prepares you for are:

- Is my kernel actually using Tensor Cores?
- Why is speed low even though Tensor Cores are enabled?
- Why does a model stop learning under reduced precision?
- Is the error coming from multiply precision, accumulation precision, or storage precision? 
    - (These are just different stages in the mathematical pipeline. Each stage can introduce a different amount of numerical loss.)

## CUDA Cores vs. Tensor Cores
*This distinction is very important because it highlights the importance of this level of programming*

To understand why your code needs to change this week, you have to understand the physical shift in NVIDIA’s hardware architecture. 

Historically, GPUs computed everything using standard **CUDA Cores**. Tensor Cores are a completely separate, specialized math engine sitting on the exact same silicon die.

### 1. Standard CUDA Cores (Scalar / Vector Processing)
* **The Mechanism:** A traditional CUDA Core is an FMA (Fused Multiply-Add) engine. It processes **one calculation at a time per thread**.
* **The Math:** If a thread needs to compute $d = (a \times b) + c$, it takes the inputs, executes the math over a clock cycle, and outputs a single scalar value. 
* **The Limitation:** To perform a massive matrix multiplication, thousands of threads have to coordinate millions of individual scalar operations, constantly reading and writing intermediate steps back and forth to registers.

### 2. Tensor Cores (Matrix Processing Engines)
* **The Mechanism:** A Tensor Core does not understand scalar numbers. It is hardwired to process **entire matrix tiles simultaneously in a single clock cycle**.
* **The Math:** Instead of computing a single number, an entire warp presents a full Matrix $A$ and Matrix $B$ to the Tensor Core. The core drops a massive hardware "stamp" down, computing the entire matrix tile product $D = (A \times B) + C$ instantaneously.

### Connecting Hardware to Your Code
This hardware layout is exactly why we cannot use standard C++ arrays or normal indexing this week:
1. Standard CUDA cores read standard variables out of standard registers.
2. **Tensor Cores require Fragments.** The specialized `wmma::fragment` containers you declare in your code are structurally engineered to distribute your matrix data across warp registers in the exact geometric pattern that the physical Tensor Core input gates expect.

If your data isn't packaged in a fragment, it cannot physically enter the Tensor Core pipeline.

## Concept Core: Warp-Level WMMA
*Before diving into Tensor Cores, let's look at a quick refresher on how the hardware physically groups your code's threads.*

**The 60-Second CUDA Hierarchy Refresher**
When you launch a GPU kernel, you specify a grid of execution. The hierarchy flows from large pools of threads down to the individual worker:

**Grid:** The entire collection of threads launched for a kernel.

**Block:** A user-defined cluster of threads that can share local memory (Shared Memory).

**Warp:** The physical reality. The GPU hardware automatically slices every Block into tight, hardware-locked teams of exactly 32 sequential threads.

In standard CUDA programming, even though the hardware schedules threads in groups of 32 (warps), you write your code from the perspective of a single independent thread using its unique ID (threadIdx.x).

WMMA (Warp Matrix Multiply and Accumulate) completely flips this paradigm. Instead of writing code for 32 independent workers, a WMMA operation is a warp contract. You program the entire 32-thread warp as if it were a single, unified entity with 32 hands.

For a hardware-standard 16×16 matrix tile, the math divides across the group beautifully:

- Tile elements: $256$
- Threads in a warp: $32$
- Elements handled per thread (conceptually): $256 / 32 = 8$

### Why all WMMA functions end in _sync:
Because the data is spread across the private registers of 32 different threads, functions like `wmma::load_matrix_sync` and `wmma::mma_sync` force all 32 threads in the warp to arrive at that instruction at the exact same clock cycle. If you introduce branch divergence (like wrapping a WMMA call inside an if (`threadIdx.x == 0`) block), the collective contract breaks, and the GPU will physically halt or produce corrupted memory states.

All WMMA functions end in `_sync` because all lanes must participate in lockstep. Divergence at this stage breaks the collective assumption.

## What is a fragment?

Students often treat `wmma::fragment` as a black box array, which leads to index bugs. 

A fragment is an **opaque C++ container** representing a sub-tile of a matrix. It does not exist in global memory or shared memory; **it sits directly in the warp's physical registers.**

Tensor cores have specialized, hardwired data routing lanes. A fragment acts as a contract between the programmer and the compiler to layout data in standard registers in a highly specific way so that these hardwired routing lanes can feed the Tensor Core hardware at maximum bandwidth. Fragments are just data structures optimized for tensor cores.

## Precision Ladder and Hardware Density

- **FP16**: 16-bit format (5 exponent, 10 mantissa).
- **TF32**: FP32-like range with truncated mantissa for Tensor Core math path.
- **FP8**: much smaller format, higher density, lower precision margin.

GEMM is usually compute-bound because data is reused across many fused multiply-add operations. Lower precision means smaller multiplier circuits, so more arithmetic units can fit on-die, increasing theoretical peak throughput.

## Implementation Milestones (Do This In Order)

1. Confirm build works and program runs once end-to-end.
2. Validate FP16 WMMA path for correctness on small test dimensions first.
3. Enable TF32 path and verify architecture target is Ampere+ (`sm_80+`).
4. Benchmark on full shape and compute TFLOPS.
5. Compare numerical error against cuBLAS FP32 reference.
6. Run precision-loss analysis and explain underflow + scaling behavior.
7. Produce a short findings report with limitations and next optimizations.

## Build and Run Commands (Student Workflow)

Run from repository root:

```bash
cmake -S . -B build -DCMAKE_CUDA_ARCHITECTURES=80
cmake --build build -j
./build/week6_tensor_cores
```

When to rerun each command:

- Re-run configure if build settings change or `build/` is recreated.
- Re-run build after source edits.
- Re-run executable whenever you need fresh benchmark/error output.

## Converting ms to TFLOPS

Your binary prints runtime in milliseconds.
You need a formula to interpret that runtime.

$$\text{TFLOPS} = \frac{2MNK}{t \cdot 10^{12}}$$

Where:
- $M, N, K$ are matrix dimensions
- $t$ is execution time in seconds

In [ ]:
def tflops(m, n, k, milliseconds):
    seconds = milliseconds / 1000.0
    return (2.0 * m * n * k) / (seconds * 1.0e12)

M, N, K = 4096, 4096, 11008
example_ms = 3.0
print(f'Example throughput at {example_ms} ms: {tflops(M, N, K, example_ms):.2f} TFLOPS')

## WMMA Kernel Mental Model (What Each Block Computes)

In `wmma_fp16.cu`, each block has one warp (`dim3 block(32)`), and each block maps to one output tile:

- `row = blockIdx.y * 16`
- `col = blockIdx.x * 16`

These are **tile start coordinates**, not global totals. The kernel then loops over K in tile-sized chunks and accumulates into one output fragment before storing to C.

In [ ]:
tile_m, tile_n = 16, 16
for by in range(2):
    for bx in range(3):
        row = by * tile_m
        col = bx * tile_n
        print(f'block ({bx},{by}) -> output tile starts at row={row}, col={col}')

## Required Experiments for Mini-Project 6

### A) Throughput
The Task: 
- Code and Run on the remote server to collect execution runtimes and compute the achieved TFLOPS for three distinct paths: cuBLAS SGEMM, WMMA TF32, and WMMA FP16 using the LLaMA-scale tensor footprint ($4096 \times 4096 \times 11008$).  

Why: 
- Writing code that uses Tensor Cores doesn't automatically make it fast. This experiment exposes the compute-vs-memory bottleneck. It forces you to look at how much performance you lose when using a naive global memory pointer strategy compared to a heavily optimized, vendor-tuned implementation (cuBLAS).  

The Industry Context: 
- In infrastructure engineering, ML compilers (like NVIDIA TensorRT, OpenAI Triton, or PyTorch Inductor) constantly have to decide whether to launch a vendor-supplied black-box library or generate a custom fused kernel. If your custom kernel only hits 15% of the vendor's speed because of memory staging issues, it will be rejected by the deployment pipeline.

### B) Numerical Error
The Task: 
- Run the 1000-trial statistical protocol comparing your custom WMMA FP16 and TF32 outputs against the FP32 ground truth. Record and summarize the distribution of maximum absolute differences, Relative $L_2$ errors and mean and percentile statistics across trials.  

Why: 
- Dropping precision bits down to 16-bit choices introduces rounding errors. This experiment teaches you that a single run is completely meaningless when measuring precision. You need to analyze the tail distribution (e.g., $99\text{th}$ percentile and worst-case deviations) to determine if rounding errors are compounding destructively over massive matrix dimensions.  

The Industry Context:
- When serving high-throughput models (like a live customer-facing chatbot processing millions of tokens per second), a single catastrophic rounding error in a deep layer can cause a model to suddenly output complete gibberish or hallucinate wildly. Companies run exhaustive statistical profiling to map out the exact "precision floor" their models can tolerate before users notice a drop in output quality.  

### C) Loss Scaling
The Task: 
- Execute analysis/precision_loss.py to simulate a toy transformer training loop. Compare and record the training loss trajectory over time under three conditions: pure FP32, pure FP16 (unscaled), and FP16 with a $1024.0$ scale factor.  

Why: 
- This bridges the gap between hardware architecture and machine learning optimization. It proves that a hardware constraint—the inability of a 16-bit floating-point container to hold microscopic decimals—has a direct, catastrophic impact on calculus and training convergence.  

The Industry Context: 
- Training a foundational model costs millions of dollars in compute time. If gradients silently underflow to absolute 0.0, the model enters a state called "silent stalling"—where GPUs are running at maximum power, consuming electricity, but the model is literally learning absolutely nothing because its weights are frozen. Understanding how to debug precision bounds prevents massive, costly cluster waste.

## 1000-Trial Accuracy Protocol (Suggested)

1. Generate 1000 random matrix pairs with fixed seed control.
2. For each trial, run reference cuBLAS FP32 and candidate WMMA path.
3. Record max-abs and relative-L2 per trial.
4. Report mean, std, p50, p90, p99, and worst-case.

This is more informative than a single run because distribution tails reveal numerical risk.

In [ ]:
import math

def summary_stats(values):
    values = sorted(values)
    n = len(values)
    def q(p):
        idx = min(n - 1, max(0, int(round((n - 1) * p))))
        return values[idx]
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / n
    return {
        'mean': mean,
        'std': math.sqrt(var),
        'p50': q(0.50),
        'p90': q(0.90),
        'p99': q(0.99),
        'max': values[-1],
    }

toy = [0.001, 0.002, 0.0008, 0.0032, 0.0011]
print(summary_stats(toy))

## Debugging Checklist (When Results Look Wrong)

- Build target architecture is Ampere+ (`sm_80`) for TF32 WMMA.
- Matrix layouts match kernel expectations (A row-major, B col-major for this implementation).
- Tile dimensions match WMMA shape constraints (FP16 `16x16x16`, TF32 `16x16x8`).
- Leading dimension passed to `load_matrix_sync` is correct.
- Grid dimensions cover full output matrix.
- Boundary guard is present for edge tiles.
- Timing uses CUDA events around kernel launch and stream sync.
- Reference and candidate outputs are copied back before error metrics.

## Stretch Goals (Optional)

- Add multi-warp threadblock tiles and shared-memory staging.
- Compare arithmetic intensity and memory traffic between naive and tiled designs.
- Add a baseline using cuBLAS Tensor Op math mode for direct comparison.
- Add a compact benchmark script to automate repeated runs and CSV output.